# Cutout from a Neuroglancer link

Download a 3D sub-volume from the zarr / precomputed sources behind a Neuroglancer
state link, then view it.


## 1. TLS certificates (internal servers only)

If your data is served behind an internal certificate authority, point Python and
tensorstore at that CA bundle before importing anything. Set `CA_BUNDLE` below to
your own `.pem` file, or skip this cell entirely for publicly trusted hosts.


In [ ]:
import os

# Path to your internal CA bundle, e.g. "~/certs/rootCA.pem".
# Leave as None if the data host uses a publicly trusted certificate.
CA_BUNDLE = os.environ.get("REQUESTS_CA_BUNDLE")

if CA_BUNDLE:
    CA_BUNDLE = os.path.expanduser(CA_BUNDLE)
    for var in ("CURL_CA_BUNDLE", "SSL_CERT_FILE", "REQUESTS_CA_BUNDLE"):
        os.environ[var] = CA_BUNDLE
    print(f"Using CA bundle: {CA_BUNDLE}")
else:
    print("No CA bundle set - using system defaults")


## 2. Download the cutout

In [ ]:
from ac_data_analysis import cutout_from_NG

# Paste a link from the Share button in Neuroglancer.
NG_LINK = "https://<neuroglancer-host>/#!https://json.neurodata.io/v1?NGStateID=<id>"

cutout, metadata = cutout_from_NG(
    ng_link=NG_LINK,
    size=(1000, 1000, 1000),   # (z, y, x) in voxels at mip 0
    mip="2",                   # note: a string, not an int
    out_path="./cutouts/",
    post_fix="example",
    save_tif=True,
)

print(f"shape {cutout.shape} (x, y, z), dtype {cutout.dtype}")
print(f"coverage {metadata['merge_info']['coverage_ratio']:.1%}")
print(f"saved to {metadata['tif_path']}")


## 3. View it

`microviewer` flips the image to match Neuroglancer's orientation. The cutout is already
transposed to x, y, z, so transpose it back before viewing.


In [ ]:
import microviewer

microviewer.view(cutout.transpose(2, 1, 0), seg=False)


## 4. Midplane

In [ ]:
import matplotlib.pyplot as plt

midplane = cutout[cutout.shape[0] // 2, :, :]
plt.imshow(midplane, cmap="gray")
plt.axis("off")
plt.show()


## Optional: contrast enhancement

```python
from skimage import exposure
img_rescale = exposure.equalize_adapthist(cutout, clip_limit=0.03)
```
